In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
race = 'TOR330'

In [3]:
TOR330_itra_2018_2019_including_DNF_df = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_itra_2018_2019_including_DNFs_df.xlsx' )
TOR330_dem = pd.read_excel(f'{race} Data/5. Clean Data for Data Visualisation/{race}_dem.xlsx' )

In [4]:
TOR330_dem = TOR330_dem[ ['Year', 'Race', 'Name', 'Sex', 'Nationality', 'Category',
       'Status', 'Status1','Duration_seconds','Finish Category']]

In [5]:
TOR330_itra_2018_2019_including_DNF_df[['Race', 'Year', 'Name', 'ITRA_Nationality', 'Sex', 'Age', 'Performance',
       'Performance_Seconds', 'Status', 'Status1']]

,Race,Year,Name,ITRA_Nationality,Sex,Age,Performance,Performance_Seconds,Status,Status1
0,TOR330,2019,Bosatelli Oliviero,ITA,M,50,3 days 00:37:13,261433.0,Finished,Finished
1,TOR330,2019,Reynolds Galen,CAN,M,35,3 days 05:06:12,277572.0,Finished,Finished
2,TOR330,2019,Lantermino Danilo,ITA,M,38,3 days 07:09:46,284986.0,Finished,Finished
3,TOR330,2019,Erwee Tiaan,RSA,M,32,3 days 08:18:24,289104.0,Finished,Finished
4,TOR330,2019,Lukas Jens,GER,M,53,3 days 13:04:04,306244.0,Finished,Finished
...,...,...,...,...,...,...,...,...,...,...
1815,TOR330,2018,Zdon Bill,USA,M,33,NaT,NaN,DNF,DNF
1816,TOR330,2018,Zennaro Davide,ITA,M,57,NaT,NaN,DNF,DNF
1817,TOR330,2018,Zimei Andrea,ITA,M,38,NaT,NaN,DNF,DNF
1818,TOR330,2018,Zimmermann Denise,SUI,F,43,NaT,NaN,DNF,DNF


In [22]:

# Create a new column 'Finish Category'
def categorize_duration(hours):
    if hours < 60:
        return 'Sub-60'
    elif hours <= 150:
        return f'{int(hours // 10) * 10}-{int(hours // 10) * 10+9}'  # Round to nearest 10 up to 150
    else:
        return 'Over-150'
    
# Define the desired order of categories
finish_category_order = [
    'Sub-60', '60-69',
    '70-79','80-89','90-99', '100-109', '110-119', '120-129',
    '130-139', '140-149',  'Over-150']
    
TOR330_itra_2018_2019_including_DNF_df  = TOR330_itra_2018_2019_including_DNF_df .rename(columns={"Performance": "Duration",
                                                    "Performance_Seconds": "Duration_seconds",
                                                    "ITRA_Nationality": "Nationality",
                                                   }) 
TOR330_itra_2018_2019_including_DNF_df ['Status'] = TOR330_itra_2018_2019_including_DNF_df ['Status'].str.replace('Finished', 'Finished at Courmayeur')
TOR330_itra_2018_2019_including_DNF_df ['Status1'] = TOR330_itra_2018_2019_including_DNF_df ['Status'].copy() 

TOR330_itra_2018_2019_including_DNF_df .loc[TOR330_itra_2018_2019_including_DNF_df ['Status1'] == 'Finished at Courmayeur', 'Status'] = 'True'
TOR330_itra_2018_2019_including_DNF_df .loc[TOR330_itra_2018_2019_including_DNF_df ['Status1'] == 'DNF', 'Status'] = 'False'

# Convert to timedelta and get total hours (handling NaT)
TOR330_itra_2018_2019_including_DNF_df ['Duration_hours'] = pd.to_timedelta(
    TOR330_itra_2018_2019_including_DNF_df ['Duration'], errors='coerce'
).dt.total_seconds() / 3600  # Convert seconds to hours



TOR330_itra_2018_2019_including_DNF_df ['Finish Category'] = TOR330_itra_2018_2019_including_DNF_df ['Duration_hours'].apply(categorize_duration)


# Set 'Finish Category' as a categorical column with the defined order
TOR330_itra_2018_2019_including_DNF_df ['Finish Category'] = pd.Categorical(
    TOR330_itra_2018_2019_including_DNF_df ['Finish Category'],
    categories = finish_category_order,
    ordered = True
)

def categorize_age(age):
    if pd.isna(age):
        return pd.nan  # Handle NaT values
    elif age < 40: # under40
        return 'SEN'
    elif age < 50: #40-49
        return f'V1'
    elif age < 60: #50-59
        return f'V2'
    elif age < 70: #60-69
        return f'V3'
    else:
        return 'V4' # Over 70

TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Status1'] == 'True', 'Status1']  = 'Finished at Courmayeur'

TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Status1'] == 'False', 'Status1']  = 'DNFs'


TOR330_itra_2018_2019_including_DNF_df ['Category'] = TOR330_itra_2018_2019_including_DNF_df['Age'].astype('int').apply(categorize_age)

TOR330_itra_2018_2019_including_DNF_df  = TOR330_itra_2018_2019_including_DNF_df [[
    'Race','Year',  'Name',  'Sex', 'Nationality','Category',
       'Status', 'Status1',  'Duration_hours', 'Duration_seconds', 'Finish Category']]

TOR330_itra_2018_2019_including_DNF_df.head()

In [7]:

# Replace '-' with NaN and convert to integer
TOR330_itra_2018_2019_including_DNF_df['Age'] = pd.to_numeric(
    TOR330_itra_2018_2019_including_DNF_df['Age'], errors='coerce'
).astype('Int64')  # Keeps NaNs as nullable integers

print(TOR330_itra_2018_2019_including_DNF_df['Age'].unique())


<IntegerArray>
[  50,   35,   38,   32,   53,   43,   31,   45,   47,   34,   39,   41,   49,
   46,   40,   27,   37,   54,   52,   36,   29,   33,   56,   44,   51,   55,
   48,   42,   58,   28,   63,   60,   62,   57,   59,   26, <NA>,   66,   61,
   30,   67,   25,   24,   21,   71,   64,   65,   22,   68,   69,   70,   73,
   23,   75]
Length: 54, dtype: Int64


In [23]:
TOR330_itra_2018_2019_including_DNF_df ['Category'] = TOR330_itra_2018_2019_including_DNF_df['Age'].apply(categorize_age)

TOR330_itra_2018_2019_including_DNF_df  = TOR330_itra_2018_2019_including_DNF_df [[
    'Race','Year',  'Name',  'Sex', 'Nationality','Category',
       'Status', 'Status1',  'Duration_hours', 'Duration_seconds', 'Finish Category']]

TOR330_itra_2018_2019_including_DNF_df.head()

TypeError: boolean value of NA is ambiguous

In [19]:
for nationality in TOR330_itra_2018_2019_including_DNF_df['Nationality'].unique():
    if len(nationality) !=2:
        print(nationality)
#         print(f'TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df[\'Nationality\'] == \'{nationality}\', \'Nationality\'] == \'\'')


 GER
 ROU
 CZE
 CHN
 CRO
 NOR
 COL
 AUS
 UKR
 HUN
 SWE
 ISL
 BUL
 ARG
 MEX
 IND
 NZL
 ECU
 BRA
 LTU
 TPE
 SRB
 GRE
 SVK
 KOR
 RUS
 AUT
 PHI
 SGP
 KEN
 HKG
 CHE
 MAC
 LBN
 THA
 INA
 MAD
 VIE
 IRL
 MDA
 CHI
 URU
 QAT
 MNE
 ESA
 ISR
 BRU
 AFG


In [11]:
# for i in sorted(list(TOR330_itra_2018_2019_including_DNF_df['Nationality'].unique())): 
#     print(f'TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df[\'Nationality\'] == \'{i}\', \'Nationality\'] == \'\'')

TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' AND', 'Nationality'] = 'AD'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' BEL', 'Nationality'] = 'BE'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CAN', 'Nationality'] = 'CA'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CRC', 'Nationality'] = 'CR'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' DEN', 'Nationality'] = 'DK'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ESP', 'Nationality'] = 'ES'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' FIN', 'Nationality'] = 'FI'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' FRA', 'Nationality'] = 'FR'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' GBR', 'Nationality'] = 'GB'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ITA', 'Nationality'] = 'IT'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' JPN', 'Nationality'] = 'JP'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' LUX', 'Nationality'] = 'LU'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MAS', 'Nationality'] = 'MY'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MON', 'Nationality'] = 'ME'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' NED', 'Nationality'] = 'NL'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' PER', 'Nationality'] = 'PE'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' POL', 'Nationality'] = 'PL'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' POR', 'Nationality'] = 'PT'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' RSA', 'Nationality'] = 'ZA'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SLO', 'Nationality'] = 'SI'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SUI', 'Nationality'] = 'CH'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' TUR', 'Nationality'] = 'TR'
TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' USA', 'Nationality'] = 'US'



# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' GER', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ROU', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CZE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CHN', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CRO', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' NOR', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' COL', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' AUS', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' UKR', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' HUN', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SWE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ISL', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' BUL', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ARG', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MEX', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' IND', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' NZL', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ECU', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' BRA', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' LTU', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' TPE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SRB', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' GRE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SVK', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' KOR', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' RUS', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' AUT', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' PHI', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' SGP', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' KEN', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' HKG', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CHE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MAC', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' LBN', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' THA', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' INA', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MAD', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' VIE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' IRL', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MDA', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' CHI', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' URU', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' QAT', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' MNE', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ESA', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' ISR', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' BRU', 'Nationality'] == ''
# TOR330_itra_2018_2019_including_DNF_df.loc[TOR330_itra_2018_2019_including_DNF_df['Nationality'] == ' AFG', 'Nationality'] == ''

In [ ]:
# Concatenate along columns (axis=1)
TOR330_dem_2018_2024 = pd.concat([TOR330_dem, TOR330_itra_2018_2019_including_DNF_df])

# Set 'Finish Category' as a categorical column with the defined order
TOR330_dem_2018_2024['Year'] = pd.Categorical(
    TOR330_dem_2018_2024['Year'],
    categories = ['2019',  '2021', '2022', '2023', '2024', ],
    ordered = True
)


TOR330_dem_2018_2024['Year'].unique()

In [ ]:
n = 0

nationality_df = []
for name in list(TOR330_dem_2018_2024['Name'].unique()):
    name_df = TOR330_dem_2018_2024[TOR330_dem_2018_2024['Name'] == name]
    if len(list(name_df['Year'].unique())) == 1:
        print(name_df['Name'].unique())
        n = n+1
        print(n, name_df['Name'].unique(), name_df['Nationality'].unique())
        nationality_df.append(name_df)
    else:
        name_df.loc[name_df['Year'] == '2019', 'Nationality'] = np.nan
        name_df['Nationality'] = name_df['Nationality'].bfill()
        nationality_df.append(name_df)
        
TOR330_dem_2018_2024_1= pd.concat(nationality_df)

TOR330_dem_2018_2024_1[TOR330_dem_2018_2024_1['Name'] == 'Papi Luca'].reset_index(drop = True)